# Install

In [1]:
import subprocess
subprocess.run(["pip", "install", "azure-storage-blob", "pyarrow", "pandas", "holidays", "pytz", "-q"])

CompletedProcess(args=['pip', 'install', 'azure-storage-blob', 'pyarrow', 'pandas', 'holidays', 'pytz', '-q'], returncode=0)

# Khởi tạo SparkSession

In [2]:
import os
import sys
from pyspark.sql import SparkSession

ACCOUNT_NAME = os.environ["AZURE_STORAGE_ACCOUNT_NAME"]
ACCOUNT_KEY  = os.environ["AZURE_STORAGE_ACCOUNT_KEY"]
CONTAINER    = os.environ.get("AZURE_CONTAINER_NAME", "nyc-traffic-lakehouse")
CONN_STR     = os.environ["AZURE_STORAGE_CONNECTION_STRING"]

spark = (SparkSession.builder
    .appName("Bronze_to_Silver")
    .master("spark://spark-master:7077")
    .config("spark.sql.session.timeZone", "America/New_York")
    .config("spark.pyspark.python", "/usr/bin/python3")
    .config("spark.executorEnv.PYSPARK_PYTHON", "/usr/bin/python3")
    .config("spark.jars.packages", 
            "org.apache.hadoop:hadoop-azure:3.3.4,"
            "io.delta:delta-spark_2.13:4.0.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config(f"spark.hadoop.fs.azure.account.key.{ACCOUNT_NAME}.blob.core.windows.net", ACCOUNT_KEY)
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")
print(f"Driver Python: {sys.executable}")

Spark version: 4.0.0
Master: spark://spark-master:7077
Driver Python: /opt/conda/bin/python


# Imports và config còn lại

In [3]:
import json
import holidays
import pytz
from datetime import date
from azure.storage.blob import BlobServiceClient
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

# Azure blob client (dùng để đọc active_link_ids.json và upload nhỏ)
blob_service = BlobServiceClient.from_connection_string(CONN_STR)

# Base path Azure Blob Storage
BASE_PATH = f"wasbs://{CONTAINER}@{ACCOUNT_NAME}.blob.core.windows.net"

eastern = pytz.timezone("America/New_York")

print(f"BASE_PATH: {BASE_PATH}")

BASE_PATH: wasbs://nyc-traffic-lakehouse@nyctrafficlakehouse.blob.core.windows.net


# Load active_link_ids + build holiday broadcast

In [4]:
# Load active_link_ids từ Azure
blob = blob_service.get_blob_client(container=CONTAINER, blob="artifacts/active_link_ids.json")
active_link_ids = json.loads(blob.download_blob().readall())
print(f"Active links: {len(active_link_ids)}")

# Build holiday set 2024–2026
us_holidays = holidays.US(subdiv="NY", years=[2024, 2025, 2026])
holiday_dates = set(str(d) for d in us_holidays.keys())  # "2024-01-01", ...

# Convert sang Spark DataFrame rồi broadcast join — không dùng UDF row-wise
holiday_df = spark.createDataFrame(
    [(d,) for d in holiday_dates], 
    ["holiday_date"]
)
holiday_broadcast = F.broadcast(holiday_df)

print(f"Holiday dates: {len(holiday_dates)} ngày (2024–2026)")
print(sorted(list(holiday_dates))[:100])

Active links: 125
Holiday dates: 43 ngày (2024–2026)
['2024-01-01', '2024-01-15', '2024-02-12', '2024-02-15', '2024-02-19', '2024-05-27', '2024-06-19', '2024-07-04', '2024-09-02', '2024-10-14', '2024-11-05', '2024-11-11', '2024-11-28', '2024-12-25', '2025-01-01', '2025-01-20', '2025-02-12', '2025-02-15', '2025-02-17', '2025-05-26', '2025-06-19', '2025-07-04', '2025-09-01', '2025-10-13', '2025-11-04', '2025-11-11', '2025-11-27', '2025-12-25', '2026-01-01', '2026-01-19', '2026-02-12', '2026-02-15', '2026-02-16', '2026-05-25', '2026-06-19', '2026-07-03', '2026-07-04', '2026-09-07', '2026-10-12', '2026-11-03', '2026-11-11', '2026-11-26', '2026-12-25']


# Hàm process Bronze → Silver

In [5]:
def process_bronze_to_silver(df):
    # 1. Cast và filter speed
    df = df.withColumn("speed", F.col("speed").cast("double"))
    df = df.filter((F.col("speed") > 0) & (F.col("speed") < 100))

    # 2. Lọc link_id active
    df = df.filter(F.col("link_id").isin(active_link_ids))

    # 3. Parse data_as_of: UTC naive string → timestamp → convert ET tường minh
    df = df.withColumn(
        "data_as_of",
        F.to_timestamp("data_as_of", "yyyy-MM-dd'T'HH:mm:ss.SSS")
    )
    # Convert tường minh UTC → America/New_York
    df = df.withColumn(
        "data_as_of",
        F.from_utc_timestamp(F.col("data_as_of"), "America/New_York")
    )

    # 4. Extract time features từ ET (session.timeZone=America/New_York đảm bảo nhất quán)
    df = df.withColumn("hour",        F.hour("data_as_of"))
    df = df.withColumn("day_of_week", F.dayofweek("data_as_of"))
    df = df.withColumn("month",       F.month("data_as_of"))
    df = df.withColumn("is_weekend",
        F.when(F.col("day_of_week").isin(1, 7), 1).otherwise(0).cast(IntegerType())
    )

    # 5. is_holiday: broadcast join theo date string
    df = df.withColumn("date_str", F.date_format("data_as_of", "yyyy-MM-dd"))
    df = df.join(holiday_broadcast, df["date_str"] == holiday_df["holiday_date"], "left")
    df = df.withColumn(
        "is_holiday",
        F.when(F.col("holiday_date").isNotNull(), 1).otherwise(0).cast(IntegerType())
    )
    df = df.drop("date_str", "holiday_date")

    # 6. Giữ đúng schema Silver
    silver_cols = ["link_id", "borough", "link_name", "data_as_of", "speed",
                   "hour", "day_of_week", "month", "is_weekend", "is_holiday"]
    return df.select(silver_cols)

print("process_bronze_to_silver() sẵn sàng")

process_bronze_to_silver() sẵn sàng


# Test thử 1 tháng trước khi chạy toàn bộ

In [ ]:
# Đọc Bronze tháng 01/2024
df_bronze_test = spark.read.parquet(
    f"{BASE_PATH}/bronze/dot_traffic_speeds/historical/year=2024/month=01/raw_2024_01.parquet"
)

print(f"Bronze rows: {df_bronze_test.count()}")
print(f"Columns: {df_bronze_test.columns}")

# Process
df_silver_test = process_bronze_to_silver(df_bronze_test)

print(f"\nSilver rows: {df_silver_test.count()}")
print(f"Link unique: {df_silver_test.select('link_id').distinct().count()}")
df_silver_test.printSchema()
df_silver_test.show(3, truncate=False)

# Ghi Silver Delta Lake + check đã upload

In [7]:
from delta.tables import DeltaTable

SILVER_PATH = f"{BASE_PATH}/silver/traffic_cleaned"

def is_silver_uploaded(year: int, month: int) -> bool:
    # Check partition đã có trong Delta chưa bằng cách đọc thử
    try:
        count = (spark.read.format("delta").load(SILVER_PATH)
                 .filter((F.col("year") == year) & (F.col("month") == month))
                 .limit(1).count())
        return count > 0
    except:
        return False

def write_silver_delta(df: "DataFrame", year: int, month: int):
    # Ghi Silver lên Azure Delta Lake
    # - Extract year/month từ data_as_of thực tế (không dùng partition gốc Bronze)
    # - Dùng append mode, partition by year + month
    # Extract year/month từ data_as_of thực tế
    df = df.withColumn("year",  F.year("data_as_of").cast("int"))
    df = df.withColumn("month", F.month("data_as_of").cast("int"))

    row_count = df.count()
    (df.write.format("delta").mode("append")
        .partitionBy("year", "month")
        .save(SILVER_PATH))
    print(f"Ghi Silver Delta {year}-{month:02d}: {row_count} rows → {SILVER_PATH}")

print("Delta Silver functions oke")

Delta Silver functions oke


# Chạy toàn bộ tháng Bronze → Silver → ghi Delta

In [8]:
from datetime import date

now = date.today()
months = []
y, m = 2024, 1
while (y, m) <= (now.year, now.month):
    months.append((y, m))
    m += 1
    if m > 12:
        m = 1
        y += 1

print(f"Tổng số tháng cần xử lý: {len(months)}")

for year, month in months:
    if is_silver_uploaded(year, month):
        print(f"Skip {year}-{month:02d} (Silver đã có)")
        continue

    print(f"\nProcessing Bronze → Silver {year}-{month:02d}...")
    
    bronze_path = f"{BASE_PATH}/bronze/dot_traffic_speeds/historical/year={year}/month={month:02d}/raw_{year}_{month:02d}.parquet"
    
    try:
        df_bronze = spark.read.parquet(bronze_path)
    except Exception as e:
        print(f"Không đọc được Bronze {year}-{month:02d}: {e}")
        continue

    df_silver = process_bronze_to_silver(df_bronze)

    if df_silver.rdd.isEmpty():
        print(f"Silver {year}-{month:02d} trống sau filter, bỏ qua")
        continue

    write_silver_delta(df_silver, year, month)

print("\nHoàn thành Bronze → Silver toàn bộ")

Tổng số tháng cần xử lý: 30

Processing Bronze → Silver 2024-01...
Ghi Silver Delta 2024-01: 801252 rows → wasbs://nyc-traffic-lakehouse@nyctrafficlakehouse.blob.core.windows.net/silver/traffic_cleaned

Processing Bronze → Silver 2024-02...
Ghi Silver Delta 2024-02: 479075 rows → wasbs://nyc-traffic-lakehouse@nyctrafficlakehouse.blob.core.windows.net/silver/traffic_cleaned

Processing Bronze → Silver 2024-03...
Ghi Silver Delta 2024-03: 829855 rows → wasbs://nyc-traffic-lakehouse@nyctrafficlakehouse.blob.core.windows.net/silver/traffic_cleaned

Processing Bronze → Silver 2024-04...
Ghi Silver Delta 2024-04: 816668 rows → wasbs://nyc-traffic-lakehouse@nyctrafficlakehouse.blob.core.windows.net/silver/traffic_cleaned

Processing Bronze → Silver 2024-05...
Ghi Silver Delta 2024-05: 827826 rows → wasbs://nyc-traffic-lakehouse@nyctrafficlakehouse.blob.core.windows.net/silver/traffic_cleaned

Processing Bronze → Silver 2024-06...
Ghi Silver Delta 2024-06: 844802 rows → wasbs://nyc-traffic-lak

# Tạo link_coordinates.parquet

In [6]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, DoubleType
from pyspark.sql.functions import udf
from datetime import date
import json

print("Tạo link_coordinates.parquet...")

# Load active_link_ids
blob_client = blob_service.get_blob_client(container=CONTAINER, blob="artifacts/active_link_ids.json")
active_link_ids = json.loads(blob_client.download_blob().readall())
print(f"Active links: {len(active_link_ids)}")

# Đọc 1 row mỗi partition tháng → union → dedupe theo link_id
dfs = []
y, m = 2024, 1
now = date.today()
while (y, m) <= (now.year, now.month):
    path = f"{BASE_PATH}/bronze/dot_traffic_speeds/historical/year={y}/month={m:02d}/"
    try:
        df_month = (spark.read.parquet(path)
                    .select("link_id", "link_points")
                    .filter(F.col("link_points").isNotNull())
                    .limit(200))
        dfs.append(df_month)
    except:
        pass
    m += 1
    if m > 12:
        m = 1
        y += 1

df_all = dfs[0]
for d in dfs[1:]:
    df_all = df_all.union(d)

# Dedupe + filter active_link_ids
df_coords = df_all.dropDuplicates(["link_id"]) \
    .filter(F.col("link_points").isNotNull()) \
    .filter(F.col("link_id").isin(active_link_ids))

# Parse điểm đầu tiên ra lat/lon
def parse_first_point(link_points):
    try:
        first = link_points.strip().split(" ")[0]
        lat, lon = first.split(",")
        return float(lat), float(lon)
    except:
        return None, None

schema = StructType([StructField("lat", DoubleType()), StructField("lon", DoubleType())])
parse_udf = udf(parse_first_point, schema)

df_coords = df_coords.withColumn("coords", parse_udf("link_points")) \
    .withColumn("lat", F.col("coords.lat")) \
    .withColumn("lon", F.col("coords.lon")) \
    .drop("coords", "link_points") \
    .filter(F.col("lat").isNotNull())

df_coords.cache()
count = df_coords.count()
print(f"Tổng link_id có tọa độ: {count}")
if count != 125:
    print(f"Kỳ vọng 125 rows, thực tế {count}")
else:
    print("Đúng 125 rows")
df_coords.show(5)

# Ghi ra artifacts/
COORDS_PATH = f"{BASE_PATH}/artifacts/link_coordinates.parquet"
df_coords.write.mode("overwrite").parquet(COORDS_PATH)
print(f"Ghi link_coordinates.parquet → {COORDS_PATH}")
df_coords.unpersist()

Tạo link_coordinates.parquet...
Active links: 125
Tổng link_id có tọa độ: 125
Đúng 125 rows
+-------+----------+----------+
|link_id|       lat|       lon|
+-------+----------+----------+
|4616324|  40.76375|-73.999191|
|4362251|40.7537504|-73.744391|
|4616192|  40.61052| -74.09769|
|4763649|  40.60414|-74.052411|
|4616310|  40.82495|-73.836211|
+-------+----------+----------+
only showing top 5 rows
Ghi link_coordinates.parquet → wasbs://nyc-traffic-lakehouse@nyctrafficlakehouse.blob.core.windows.net/artifacts/link_coordinates.parquet


DataFrame[link_id: string, lat: double, lon: double]

# Verify

In [7]:
df_silver_check = spark.read.format("delta").load(
    f"{BASE_PATH}/silver/traffic_cleaned"
)
print(f"Tổng rows Silver: {df_silver_check.count()}")
print(f"Link unique: {df_silver_check.select('link_id').distinct().count()}")
print(f"Tháng có: {df_silver_check.select(F.year('data_as_of').alias('year'), F.month('data_as_of').alias('month')).distinct().orderBy('year','month').collect()}")
df_silver_check.show(3)

Tổng rows Silver: 23522760
Link unique: 116
Tháng có: [Row(year=2023, month=12), Row(year=2024, month=1), Row(year=2024, month=2), Row(year=2024, month=3), Row(year=2024, month=4), Row(year=2024, month=5), Row(year=2024, month=6), Row(year=2024, month=7), Row(year=2024, month=8), Row(year=2024, month=9), Row(year=2024, month=10), Row(year=2024, month=11), Row(year=2024, month=12), Row(year=2025, month=1), Row(year=2025, month=2), Row(year=2025, month=3), Row(year=2025, month=4), Row(year=2025, month=5), Row(year=2025, month=6), Row(year=2025, month=7), Row(year=2025, month=8), Row(year=2025, month=9), Row(year=2025, month=10), Row(year=2025, month=11), Row(year=2025, month=12), Row(year=2026, month=1), Row(year=2026, month=2), Row(year=2026, month=3), Row(year=2026, month=4), Row(year=2026, month=5), Row(year=2026, month=6)]
+-------+-------------+--------------------+-------------------+-----+----+-----------+-----+----------+----------+----+
|link_id|      borough|           link_nam

# Kiểm tra sự phân bố

In [9]:
from pyspark.sql import functions as F

df_silver_all = spark.read.format("delta").load(f"{BASE_PATH}/silver/traffic_cleaned")

print(f"Tổng rows: {df_silver_all.count()}")
print(f"Link unique: {df_silver_all.select('link_id').distinct().count()}")

# Phân bố theo year/month
print("\nPhân bố theo year/month")
df_silver_all.groupBy(
    F.year("data_as_of").alias("year"),
    F.month("data_as_of").alias("month")
).count().orderBy("year", "month").show(40)

# Phân bố theo day_of_week
print("\nPhân bố theo day_of_week")
df_silver_all.groupBy("day_of_week").count().orderBy("day_of_week").show()

# Phân bố theo hour
print("\nPhân bố theo hour")
df_silver_all.groupBy("hour").count().orderBy("hour").show(24)

Tổng rows: 23522760
Link unique: 116

Phân bố theo year/month
+----+-----+------+
|year|month| count|
+----+-----+------+
|2023|   12|  5775|
|2024|    1|795477|
|2024|    2|484441|
|2024|    3|828959|
|2024|    4|816293|
|2024|    5|828270|
|2024|    6|844736|
|2024|    7|837882|
|2024|    8|808275|
|2024|    9|844721|
|2024|   10|860866|
|2024|   11|836534|
|2024|   12|703341|
|2025|    1|867296|
|2025|    2|730696|
|2025|    3|834599|
|2025|    4|805176|
|2025|    5|806033|
|2025|    6|777804|
|2025|    7|853259|
|2025|    8|889744|
|2025|    9|859711|
|2025|   10|871456|
|2025|   11|800694|
|2025|   12|815587|
|2026|    1|868362|
|2026|    2|741018|
|2026|    3|693375|
|2026|    4|538685|
|2026|    5|725643|
|2026|    6|548052|
+----+-----+------+


Phân bố theo day_of_week
+-----------+-------+
|day_of_week|  count|
+-----------+-------+
|          1|3315725|
|          2|3347808|
|          3|3328746|
|          4|3378154|
|          5|3371785|
|          6|3391917|
|          7|